In [ ]:
#| hide
from programming_first_principles.core import prepare_notebook
from collections.abc import Sequence
import sys
import unicodedata

In [ ]:
#| hide
prepare_notebook()

# String (`str`)
* A text string is a finite, ordered, immutable sequence of abstract characters, where each character is identified by its integer code point in the Unicode standard.
* `str` is the immutable sequence of Unicode code points.

## Code Points and Characters
* A **character** is the smallest abstract component of text (letters, digits, punctuation, symbols).
* A **code point** is the non-negative integer assigned to a character by the Unicode standard (`0` to `0x10FFFF`, or `1,114,111`).
* `ord(ch)` maps a single-character string to its integer code point; `chr(n)` maps the integer code point back to the single-character string.

#### `ord`, `chr`, and `sys.maxunicode`

In [ ]:
f'{ord("A") = }, {ord("a") = }, {ord("A") - ord("a") = }'

'ord("A") = 65, ord("a") = 97, ord("A") - ord("a") = -32'

In [ ]:
f'{ord("🐍") = }, {hex(ord("🐍")) = }'

'ord("🐍") = 128013, hex(ord("🐍")) = \'0x1f40d\''

In [ ]:
f'{chr(65) = }, {chr(0x1f40d) = }'

"chr(65) = 'A', chr(0x1f40d) = '🐍'"

In [ ]:
f'{sys.maxunicode = }, {hex(sys.maxunicode) = }'

"sys.maxunicode = 1114111, hex(sys.maxunicode) = '0x10ffff'"

#### Escape Sequences and Character Names

* `\xhh` — 2-digit hex code point (`0x00`..`0xFF`)


In [ ]:
f'{"\x41" = }'

'"\\x41" = \'A\''

* `\uhhhh` — 4-digit hex code point

In [ ]:
f'{"\u0041" = }'

'"\\u0041" = \'A\''

* `\Uhhhhhhhh` — 8-digit hex code point

In [ ]:
f'{"\U0001f642" = }, {"\U0001f620" = }, {"\U0001f600" = }'

'"\\U0001f642" = \'🙂\', "\\U0001f620" = \'😠\', "\\U0001f600" = \'😀\''

* `\N{name}` — character by its official Unicode name (`unicodedata.name`)

In [ ]:
f'{"\N{ANGRY FACE}" = }'

'"\\N{ANGRY FACE}" = \'😠\''

In [ ]:
f'{unicodedata.name("😆") = }'
f'{unicodedata.name("😠") = }'
f'{unicodedata.name("😊") = }'

'unicodedata.name("😆") = \'SMILING FACE WITH OPEN MOUTH AND TIGHTLY-CLOSED EYES\''

'unicodedata.name("😠") = \'ANGRY FACE\''

'unicodedata.name("😊") = \'SMILING FACE WITH SMILING EYES\''

## `str` as a Sequence
* `str` implements the `Sequence` protocol (`__len__`, `__getitem__`).
* Each position `i` holds a length-1 `str` representing one Unicode code point.

#### Literals and `str()` Constructor

In [ ]:
emoji_string = 'Python 3 - 🐍 🐍 🐍 😊 😠 😆'
constructor_str = str('Python 3 - 🐍 🐍 🐍 😊 😠 😆')

f'{emoji_string = }, {type(emoji_string) = }'
f'{emoji_string == constructor_str = }'

"emoji_string = 'Python 3 - 🐍 🐍 🐍 😊 😠 😆', type(emoji_string) = <class 'str'>"

'emoji_string == constructor_str = True'

In [ ]:
f'{isinstance(emoji_string, Sequence) = }'
f'{issubclass(str, Sequence) = }'

'isinstance(emoji_string, Sequence) = True'

'issubclass(str, Sequence) = True'

#### Subscription and Slicing

In [ ]:
f'{len(emoji_string) = }'
f'{emoji_string[0] = }, {emoji_string[11] = }, {emoji_string[19] = }'
f'{ord(emoji_string[19]) = }, U+{ord(emoji_string[19]):x}'
f'{emoji_string[:8] = }, {emoji_string[11:] = }'

'len(emoji_string) = 22'

"emoji_string[0] = 'P', emoji_string[11] = '🐍', emoji_string[19] = '😠'"

'ord(emoji_string[19]) = 128544, U+1f620'

"emoji_string[:8] = 'Python 3', emoji_string[11:] = '🐍 🐍 🐍 😊 😠 😆'"

## Immutability and Concatenation
* Like `tuple`, `str` does not support item assignment (`__setitem__`).
* Concatenation (`+` / `__add__`) allocates and returns a new `str` at a new memory address, leaving the original strings unchanged.

#### Item Assignment Fails

In [ ]:
try:
    emoji_string[0] = 'p'
except TypeError as e:
    f'{e = }'

'e = TypeError("\'str\' object does not support item assignment")'

#### Concatenation Creates a New Object

In [ ]:
s1 = emoji_string
s2 = emoji_string + ' ABC'

f'{id(emoji_string) = }, {id(s1) = }, {s1 is emoji_string = }'
f'{id(s2) = }, {s2 is emoji_string = }'
f'{s2 = }'

'id(emoji_string) = 4468120816, id(s1) = 4468120816, s1 is emoji_string = True'

'id(s2) = 4467189424, s2 is emoji_string = False'

"s2 = 'Python 3 - 🐍 🐍 🐍 😊 😠 😆 ABC'"

## Internal Code-Point Width (PEP 393)
* Under PEP 393 (Martin von Löwis), CPython stores a `str` as a fixed-width array of code points whose element width (1, 2, or 4 bytes per character) is chosen from the maximum code point in the string:
  * `<= 0xFF` (ASCII / Latin-1): 1 byte per code point
  * `<= 0xFFFF` (BMP): 2 bytes per code point
  * `<= 0x10FFFF` (Supplementary planes, e.g., emojis): 4 bytes per code point
* This keeps indexing `s[i]` $O(1)$ (`base + i * char_width`) while compacting strings that only use smaller code points.

In [ ]:
s_ascii = 'a'
s_emoji = 'a😊'
emoji_escape_str = 'Python 3 - \U0001f40d \U0001f40d \U0001f40d \U0001f60a \U0001f620 \U0001f606'

f'{len(s_ascii) = }, {sys.getsizeof(s_ascii) = }'
f'{len(s_emoji) = }, {sys.getsizeof(s_emoji) = }'
f'{emoji_string == emoji_escape_str = }'
f'{sys.getsizeof(emoji_string) = }, {sys.getsizeof(emoji_escape_str) = }'

'len(s_ascii) = 1, sys.getsizeof(s_ascii) = 42'

'len(s_emoji) = 2, sys.getsizeof(s_emoji) = 68'

'emoji_string == emoji_escape_str = True'

'sys.getsizeof(emoji_string) = 148, sys.getsizeof(emoji_escape_str) = 148'

## References
* [PEP 393 – Flexible String Representation (Martin von Löwis)](https://peps.python.org/pep-0393/)
* [Python Library Reference – Text Sequence Type `str`](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str)
* [The Unicode Standard](https://www.unicode.org/versions/latest/)